# 1. Les bases des réseaux de neurones récurrents

**Objectifs**

- Comprendre les capacités des RNN (*Recurrent Neural Networks*, réseaux de neurones récurrents) à partir de problèmes simples ;
- Se familiariser avec les outils de `torch` pour les réseaux récurrents.

**Ressources et références**

Les LSTM (*Long Short-Term Memory*) sont des réseaux récurrents dotés de mécanismes de mémoire et de portes.

- [Fiche de synthèse de Stanford sur les RNN](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-recurrent-neural-networks#overview)
- Andrej Karpathy, [« The Unreasonable Effectiveness of Recurrent Neural Networks »](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)
- J. Brownlee, [« 5 Examples of Simple Sequence Prediction Problems for LSTMs »](https://machinelearningmastery.com/sequence-prediction-problems-learning-lstm-recurrent-neural-networks/)
- [Le billet de Christopher Olah sur les LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) ; ses autres billets sont également intéressants.
- [Une démonstration interactive pour explorer les LSTM](https://distill.pub/2016/handwriting/) et aller plus loin.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

## Sommaire

- **1. Utiliser les RNN avec PyTorch**
  - *Le RNN d’Elman*
  - *Exécuter un RNN*
  - *Que se passe-t-il à l’intérieur ?*
- **2. Typologie des tâches**
  - *Vue d’ensemble*
  - *2.1. Problème simple : plusieurs entrées, une sortie*
  - *2.2. Problème simple : une entrée, plusieurs sorties*
- **3. Projet : synthèse d’écriture manuscrite**


## 1. Utiliser les RNN

Les RNN diffèrent légèrement des réseaux à propagation avant (*feedforward*), dont vous connaissez une forme courante : les perceptrons multicouches.

Dans un réseau à propagation avant, une fonction $f_{w,b}$ s’applique aux entrées $x$ pour calculer la sortie $y$ :

<p align="center">
  <img src="./static/perceptron.png" />
</p>

$$
y = f_{w,b}(x) = act(\sum_i w_i x_i + b)
$$

$w$ et $b$ sont les paramètres du réseau : les poids des connexions entre les neurones et leurs entrées, et un biais constant. $act$ désigne la fonction d’activation.

Un RNN reçoit l’entrée courante $x$, mais aussi le résultat de certains calculs précédents, noté $h$. Dans le cas le plus simple, $h$ correspond à la sortie $y$ de notre perceptron. On l’appelle **l’état interne ou état caché**.

$$
y_t = h_t = act(\sum_i w^{in}_i x_{t, i} + \sum_j w^h h_{t-1, j} + b)
$$

<p align="center">
  <img src="./static/rnn.png" />
</p>

Remarquez l’ajout des indices $t$ : **le temps intervient désormais dans le calcul**. Les observations $x$ ne sont plus traitées indépendamment les unes des autres ; elles forment des **séquences**, comme nous l’avons vu au chapitre précédent.

Un RNN simple peut se voir comme un perceptron doté d’une entrée supplémentaire jouant le rôle de mémoire. On peut donc le représenter comme l’application répétée d’un perceptron à deux entrées : $x_t$ et $h_{t-1}$.

<p align="center">
  <img src="./static/rnn-unfold.png" />
</p>

Cette représentation est appelée **dépliage temporel** : la récurrence et les pas de temps apparaissent sur l’axe horizontal.

Le perceptron s’applique d’abord au pas $t$ à $x_t$ et à $h_{t-1}$, son état interne précédent. Deux ensembles de poids, $w_{in}$ et $w_h$, paramètrent les deux types d’entrée. Une fois le nouvel état $h_t$ calculé, on passe à l’entrée suivante $x_{t+1}$, et ainsi de suite.


#### Le RNN d’Elman

Le neurone que nous venons de définir constitue l’élément de base des **réseaux d’Elman**, nommés d’après Jeffrey Locke Elman.

Ces réseaux, aussi appelés « RNN simples » ou simplement « RNN », ajoutent généralement des poids de sortie pour calculer $y_t$ à partir de $h_t$. Cela revient à ajouter une couche de connexions à partir de l’état interne $h_t$ pour réaliser la tâche souhaitée : prévision, classification, etc.

Les RNN simples sont disponibles dans PyTorch avec la classe [`RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html). Cette classe ne fournit pas de couche de sortie adaptée à votre tâche. Il vous revient d’ajouter les calculs nécessaires à partir de certains états $h_t$.

Pour commencer, observons le fonctionnement de ces réseaux sans leur demander de réaliser une tâche particulière.


In [ ]:
# Créer un RNN de 100 neurones recevant des entrées de dimension 1
rnn = ...

In [ ]:
print("Forme des poids Wh :", rnn.weight_hh_l0.shape)
print("Forme des poids Win :", rnn.weight_ih_l0.shape)

#### Exécuter un RNN

Essayons le RNN sur des données simulées : deux sinusoïdes successives de fréquences différentes.


In [ ]:
wave1 = np.sin(np.linspace(0, 4*np.pi, 100, endpoint=False))  # Deux sinusoïdes
wave2 = np.sin(np.linspace(4*np.pi, 8*np.pi, 100)*5)

wave = np.concatenate([wave1, wave2])

plt.plot(wave)
plt.show()

In [ ]:
# Toujours convertir les tableaux en torch.Tensor !
x = torch.Tensor(wave)

# Entrées : (longueur de séquence, dimension d’entrée), ou (taille du lot, longueur de séquence, dimension d’entrée).
# Ici, passer de (200,) à (200, 1), ou à (1, 200, 1) selon la convention choisie.
x =  ...

In [ ]:
with torch.no_grad():  # Désactiver le calcul automatique du gradient : il est inutile ici.
    # Appliquer le RNN à la sinusoïde. Que représentent les sorties ?
    all_outputs, last_output = ...

print(all_outputs.size(), "\n", last_output.size())
print(all_outputs, "\n", last_output)

#### Que se passe-t-il à l’intérieur ?

Les vecteurs de dimension 100 représentent les états cachés de nos 100 neurones à chaque pas de la séquence d’entrée.

Observons ces états cachés.


In [ ]:
# Tracer l’activité de quelques neurones du RNN choisis au hasard.
...
# L’activité de tous les neurones à un pas t forme l’« état caché » (hidden_state).

## Typologie des tâches


### Vue d’ensemble : quelles tâches peut-on réaliser ?

On peut utiliser les RNN de différentes façons dans une architecture de réseau de neurones, en transformant des séquences en vecteurs ou des vecteurs en séquences.

Par exemple, une image peut servir à générer une légende composée d’une suite de mots. À l’inverse, une suite de mots peut être résumée par une seule étiquette décrivant le sentiment global d’une phrase : « heureux » ou « triste ». On peut également traduire une phrase française de longueur $n$ en une phrase anglaise de longueur $m$.

Les RNN permettent donc de traiter l’axe temporel de plusieurs manières selon la tâche.

Dans les figures suivantes, les vecteurs verts $x$ représentent les entrées, le vecteur $a$ représente les états cachés, et les vecteurs rouges $y$ représentent les sorties du réseau. Ces sorties peuvent être égales à $a$ ou résulter d’un calcul supplémentaire, par exemple une couche de neurones produisant des probabilités de classes. Les RNN sont dépliés dans le temps pour rendre les séquences visibles.

Ces usages se répartissent en quatre grandes catégories.

- **Plusieurs entrées, plusieurs sorties de même longueur** (*many-to-many*) : une séquence ou une série temporelle produit une autre séquence de **même longueur**. Exemple : reconnaissance vocale au fil du signal.
<p align="center">
  <img src="./static/rnn-many-to-many-same-ltr.png" width=500/>
</p>

- **Plusieurs entrées, une sortie**, ou **séquence vers vecteur** (*many-to-one*, *seq-to-vec*) : une séquence produit un seul vecteur. Exemple : classification d’une séquence, comme l’analyse du sentiment d’une phrase.
<p align="center">
  <img src="./static/rnn-many-to-one-ltr.png" width=500/>
</p>

- **Une entrée, plusieurs sorties**, ou **vecteur vers séquence** (*one-to-many*, *vec-to-seq*) : un seul vecteur produit une séquence. Exemples : génération d’une légende d’image ou de musique.
<p align="center">
  <img src="./static/rnn-one-to-many-ltr.png" width=500/>
</p>

- **Encodeur-décodeur** : une séquence ou une série temporelle produit une autre séquence de **longueur différente**. Exemple : traduction automatique.
<p align="center">
  <img src="./static/rnn-many-to-many-different-ltr.png" width=500/>
</p>


### Plusieurs entrées, une sortie : un réseau calculatrice

Concevons un réseau capable de réaliser une **classification simple** : *déterminer si une expression arithmétique donne un résultat positif ou négatif*.

Les entrées sont des séquences construites ainsi :

- Chaque pas contient deux nombres : un opérande et un opérateur.
- Le premier nombre, l’opérande, est la valeur sur laquelle porte l’opération.
- Le second, l’opérateur, vaut 1 ou 0 : 1 signifie ajouter l’opérande au total, 0 signifie le soustraire.

L’expression suivante :

$$+ 1 + 2 - 5 + 4 - 6$$

est donc représentée par cette séquence d’entrée :

```
[[1, 1],
 [2, 1],
 [5, 0],
 [4, 1],
 [6, 0]]
```

La classe cible correspond au signe du résultat : ici, −4, donc la **classe négative**.

Toutes les séquences contiennent 5 opérandes, chacun compris entre 0 et 9.


In [ ]:
def make_operand_sequences(n_sequences, seq_length=5):
    digits = list(range(0, 10))
    operators = [0, 1]
    op_symbols = ["-", "+"]
    x = np.zeros((n_sequences, seq_length, 2))
    y = np.zeros(n_sequences)
    for i in range(n_sequences):
        operands = np.random.choice(digits, seq_length, replace=True)
        ops = np.random.choice(operators, seq_length, replace=True)
        seq = np.vstack([operands, ops])
        x[i] = seq.T
        expr = ""
        for digit, op in zip(operands, ops):
            expr += op_symbols[operators.index(op)] + str(digit)
        total = eval(expr)
        y[i] = 1 if total > 0 else 0
    return x, y

In [ ]:
x, y = make_operand_sequences(1000)

In [ ]:
class Calculator(torch.nn.Module):

    def __init__(self, hidden_size):
        super().__init__()
        
        # Les objets nn.Module peuvent contenir les opérations nécessaires
        # grâce à vos propres méthodes et attributs.
        # PyTorch offre beaucoup de souplesse.
        # Vous pouvez y stocker ce dont vous avez besoin.
        
        # Nombre de neurones du RNN
        self.hidden_size = hidden_size
        
        # Créer un RNN adapté à la tâche,
        # sans oublier batch_first=True,
        # puis une couche de sortie pour la classification.
        self.rnn = ...
        self.fc = ...

        # Ne pas ajouter pour l’instant d’activation comme softmax à la sortie.
        # Avec l’entropie croisée, PyTorch utilise des scores non normalisés
        # pour entraîner le classifieur (voir la documentation).

    def forward(self, x):
        # Effectuer le parcours avant, c’est-à-dire la prédiction.
        # Prendre le dernier état de la séquence et le transmettre à la couche de classification.
        # Cet état représente l’information accumulée
        # au fil de la séquence d’entrée.
        
        ...
        
        output = ...
        
        return output

In [ ]:
device = "cpu"  # Remplacer par "cuda" si un processeur graphique est disponible.

network = Calculator(hidden_size=100).to(device)

print(network)

# Boucle d’entraînement
num_epochs = 50  # Modifier le nombre d’époques si nécessaire.

network.train()  # Activer le mode entraînement.

# Perte : entropie croisée, car il s’agit d’une classification.
criterion = torch.nn.CrossEntropyLoss()

# Règle d’apprentissage : l’optimiseur Adam, une variante
# de la descente du gradient.
opt = torch.optim.Adam(network.parameters(), lr=1e-3)


# Convertir les données en torch.Tensor, puis créer un Dataset
# et un DataLoader.
# Ces objets gèrent les données pendant l’entraînement et permettent
# notamment de définir la taille des lots ou des transformations.
# Ici, un TensorDataset, qui reçoit des tenseurs,
# et un DataLoader avec la taille de lot choisie suffisent.
batch_size = 10  # Modifier la taille du lot si nécessaire.

x_train = torch.Tensor(x).to(device)
y_train = torch.Tensor(y).to(torch.long).to(device)  # Les étiquettes de classes doivent avoir le type torch.long.

# Créer le TensorDataset et le DataLoader.
dataset = torch.utils.data.TensorDataset(x_train, y_train)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)

losses = []
for e in range(num_epochs):

    epoch_loss = []

    for x_i, y_i in dataloader:  # Pour chaque lot de données
        
        # Boucle d’entraînement à compléter
        ...
        
        loss = ...

        # Conserver les pertes pour tracer la courbe d’apprentissage.
        loss = loss.detach().numpy()
        epoch_loss.append(loss)
        
        print(f"Époque {e} - Perte {np.mean(epoch_loss):.3f}", end="\r")

    losses.append(np.mean(epoch_loss))

In [ ]:
plt.plot(losses)

In [ ]:
x_test, y_test = make_operand_sequences(100)

In [ ]:
# Tester le réseau.
network.eval()
with torch.no_grad():
    y_pred = network(torch.Tensor(x_test))

In [ ]:
print("Exactitude :", np.sum(np.argmax(y_pred.numpy(), axis=1) == y_test) / 100)

### Une entrée, plusieurs sorties : un oscillateur

Définir une tâche de type « une entrée, plusieurs sorties » demande de bien réfléchir à l’initialisation. Dans cet exemple, pouvons-nous entraîner un RNN à générer une sinusoïde — plusieurs sorties — à partir de sa fréquence — une seule entrée ?

Pour générer une séquence à partir d’un vecteur, il faut deux éléments, comme dans une récurrence mathématique :

- Une condition initiale adaptée ;
- Une relation de récurrence entre les sorties du RNN.

<p align="center">
  <img src="./static/rnn-one-to-many-ltr.png" width=500/>
</p>

La condition initiale est le vecteur vert $x$ de la figure, c’est-à-dire l’entrée unique. Elle place le réseau dans un état à partir duquel il pourra générer toute la séquence. Ici, $x$ peut être un vecteur à codage **one-hot** représentant la fréquence souhaitée : une composante vaut 1, les autres valent 0.

On peut fournir $x$ comme première entrée, puis donner au réseau ses propres sorties précédentes. Ce mécanisme, visible dans le schéma, est appelé **injection préalable** (*pre-inject*).

D’autres mécanismes existent. On peut utiliser $x$ pour construire directement le premier état caché : **injection dans l’état initial** (*init-inject*). On peut aussi ajouter $x$ aux sorties à chaque pas, comme un rappel permanent de l’objectif : **injection à chaque pas** (*par-inject*).

Après l’initialisation, le réseau utilise ses propres sorties pour générer la séquence par récurrence.

Dans cet exemple, nous utiliserons une injection dans l’état initial. Nous projetterons le vecteur de fréquence de dimension 3 dans l’espace des états cachés. Le résultat servira à initialiser le premier état caché du réseau, nul par défaut. Une couche de connexions transforme ainsi le vecteur de fréquence pour qu’il ait la même dimension que l’état caché.

Le réseau prédit ensuite un premier point de la sinusoïde. Ce point devient l’entrée suivante, et ainsi de suite.


In [ ]:
def make_waves(n_sequences, frequencies, length):
    x = np.zeros((n_sequences, 1, len(frequencies)))
    y = np.zeros((n_sequences, length, 1))
    for i in range(n_sequences):
        fs = np.random.choice(frequencies)
        waves = np.sin(2*np.pi*fs*np.linspace(0, 1, length))
        fs_idx = frequencies.index(fs)
        x[i, 0, fs_idx] = 1  # Codage one-hot : 1 pour la fréquence choisie, 0 pour les autres.
        y[i, :, 0] = waves
    
    # Forme de y renvoyé : (n, length, 1)
    return x, y

In [ ]:
frequencies = [1, 5, 10]  # Valeurs des fréquences
length = 50  # Longueur de la sinusoïde

x, y = make_waves(1000, frequencies, length)

In [ ]:
# Forme : [nombre de séquences, longueur, dimension]
# La longueur vaut ici 1 : on fournit une seule entrée au réseau.
# Un x par sinusoïde représente la classe de fréquence.
print(x.shape, "\n", x)

In [ ]:
class SineWaveGenerator(torch.nn.Module):

    def __init__(self, hidden_size, n_frequencies, length):
        super().__init__()

        self.hidden_size = hidden_size
        self.length = length

        # Créer une couche transformant la fréquence en état caché.
        self.input_to_hidden = ...
        # Créer un RNN.
        self.rnn = ...
        # Créer une couche de sortie transformant l’état caché en un point de dimension 1.
        self.hidden_to_output = ...

    def forward(self, freq):
                
        # Stocker les sorties.
        outputs = torch.zeros(x.size(0), self.length, 1).to(freq.device)
        
        # Attention aux formes :
        # Vérifier les dimensions des sorties et de l’état caché,
        # puis transposer les tenseurs si nécessaire.

        # Injection dans l’état initial (init-inject)
        # Passer du vecteur d’entrée [n, 1, nombre de fréquences]
        # à l’état caché initial [1, n, hidden_size].
        
        hidden = ...
        
        # La première entrée est nulle.
        # Zéros de forme [n, longueur=1, dimension]
        x = ...
        
        # Générer la sinusoïde par récurrence.
        for i in range(0, self.length):
                
            y = ...
            
            # Stocker les sorties.
            outputs[:, i, 0] = y.squeeze()
            
            # Utiliser la sortie comme entrée suivante.
            x = y
            
        return outputs

In [ ]:
network = SineWaveGenerator(hidden_size=100, n_frequencies=3, length=length)

print(network)

# Boucle d’entraînement
num_epochs = 10
batch_size = 10

losses = []

# À vous de compléter l’entraînement.

In [ ]:
plt.plot(losses)

In [ ]:
x_test, y_test = make_waves(100, frequencies, length)

In [ ]:
network.eval()
with torch.no_grad():
    y_pred = network(torch.Tensor(x_test))

In [ ]:
print("Erreur quadratique moyenne (MSE) :", np.mean((y_test - y_pred.numpy())**2))

# Application aux séries temporelles non stationnaires

## 1. Le jeu de données des passagers aériens

Nous allons entraîner un RNN à prévoir une série temporelle non stationnaire, en commençant par le jeu de données connu sous le nom d’« Airline Passengers ».


In [ ]:
import pandas as pd
import urllib.request   

# Télécharger le jeu de données avec urllib.
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
filename = "airline-passengers.csv"

try:
    urllib.request.urlretrieve(url, filename)
    print(f"Jeu de données téléchargé dans '{filename}'")
    
    # Charger les données et afficher un aperçu.
    data = pd.read_csv(filename)
    print(f"\nDimensions du jeu de données : {data.shape}")
    print(f"Colonnes : {data.columns.tolist()}")
    print("\nPremières lignes :")
    print(data.head())
    
    # Tracer la série temporelle.
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 6))
    plt.plot(data.iloc[:, 1], linewidth=2)
    plt.title('Série temporelle des passagers aériens')
    plt.xlabel('Temps (mois)')
    plt.ylabel('Nombre de passagers')
    plt.grid(True, alpha=0.3)
    plt.show()
    
except Exception as e:
    print(f"Erreur lors du téléchargement : {e}")
    print("Vous pouvez télécharger les données manuellement à cette adresse : https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv")

In [ ]:
# Préparer les données pour l’entraînement du RNN.
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Extraire le nombre de passagers, dans la deuxième colonne.
passengers = data.iloc[:, 1].values.astype(float)

# Normaliser les données.
scaler = MinMaxScaler()
passengers_normalized = scaler.fit_transform(passengers.reshape(-1, 1)).flatten()

# Créer des séquences pour le RNN.
def create_sequences(data, sequence_length):
    """Créer des séquences entrée-cible pour prévoir une série temporelle."""
    X, y = [], []
    for i in range(sequence_length, len(data)):
        X.append(data[i-sequence_length:i])
        y.append(data[i])
    return np.array(X), np.array(y)

# Paramètres
sequence_length = 12  # Utiliser 12 mois pour prévoir le mois suivant.
test_size = 0.2

# Créer les séquences.
X, y = create_sequences(passengers_normalized, sequence_length)

# Séparer les données d’entraînement et de test.
split_index = int(len(X) * (1 - test_size))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Adapter la forme pour PyTorch : (exemples, longueur de séquence, variables).
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Convertir les données en tenseurs PyTorch.
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

print(f"Dimensions des données d’entraînement : X={X_train_tensor.shape}, y={y_train_tensor.shape}")
print(f"Dimensions des données de test : X={X_test_tensor.shape}, y={y_test_tensor.shape}")

# Définir le modèle RNN pour la prévision de série temporelle.
class TimeSeriesRNN(torch.nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1, output_size=1):
        super(TimeSeriesRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = torch.nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # Initialiser l’état caché.
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        # Effectuer le parcours avant dans le RNN.
        out, _ = self.rnn(x, h0)
        
        # Récupérer la sortie du dernier pas de temps.
        out = self.fc(out[:, -1, :])
        return out

# Créer le modèle.
model = TimeSeriesRNN(input_size=1, hidden_size=50, num_layers=2)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("\n Modèle créé !")
print(model)

In [ ]:
# Entraîner le modèle RNN.
num_epochs = 200
train_losses = []

print("Entraînement du modèle RNN…")

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # Parcours avant
    outputs = model(X_train_tensor)
    loss = criterion(outputs.squeeze(), y_train_tensor)
    
    # Parcours arrière
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    if (epoch + 1) % 40 == 0:
        print(f'Époque [{epoch+1}/{num_epochs}], perte : {loss.item():.6f}')

print("Entraînement terminé !")

# Tracer la perte d’entraînement.
plt.figure(figsize=(10, 6))
plt.plot(train_losses, linewidth=2, color='blue')
plt.title('Évolution de la perte d’entraînement')
plt.xlabel('Époques')
plt.ylabel('Perte : erreur quadratique moyenne')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Évaluer le modèle et visualiser les prévisions.
model.eval()

with torch.no_grad():
    # Prévisions sur les données de test
    test_predictions = model(X_test_tensor).squeeze().numpy()
    
    # Prévisions sur les données d’entraînement, pour comparaison
    train_predictions = model(X_train_tensor).squeeze().numpy()

# Revenir aux unités initiales pour les prévisions et les observations.
test_predictions_denorm = scaler.inverse_transform(test_predictions.reshape(-1, 1)).flatten()
test_actual_denorm = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
train_predictions_denorm = scaler.inverse_transform(train_predictions.reshape(-1, 1)).flatten()
train_actual_denorm = scaler.inverse_transform(y_train.reshape(-1, 1)).flatten()

# Calculer les indicateurs d’erreur.
from sklearn.metrics import mean_squared_error, mean_absolute_error
test_mse = mean_squared_error(test_actual_denorm, test_predictions_denorm)
test_mae = mean_absolute_error(test_actual_denorm, test_predictions_denorm)

print(f"Performances sur le test :")
print(f"MSE (erreur quadratique moyenne) : {test_mse:.2f}")
print(f"MAE (erreur absolue moyenne) : {test_mae:.2f}")
print(f"RMSE (racine de l’erreur quadratique moyenne) : {np.sqrt(test_mse):.2f}")

# Visualiser les prévisions.
plt.figure(figsize=(15, 8))

# Tracer les données et prévisions d’entraînement.
train_time = range(sequence_length, sequence_length + len(train_actual_denorm))
plt.plot(train_time, train_actual_denorm, 'b-', label='Observations (entraînement)', alpha=0.7)
plt.plot(train_time, train_predictions_denorm, 'g-', label='Prévisions (entraînement)', alpha=0.7)

# Tracer les données et prévisions de test.
test_time = range(sequence_length + len(train_actual_denorm), 
                 sequence_length + len(train_actual_denorm) + len(test_actual_denorm))
plt.plot(test_time, test_actual_denorm, 'b-', label='Observations (test)', linewidth=2)
plt.plot(test_time, test_predictions_denorm, 'r-', label='Prévisions (test)', linewidth=2)

# Ajouter une ligne verticale séparant entraînement et test.
plt.axvline(x=sequence_length + len(train_actual_denorm), color='black', 
           linestyle='--', alpha=0.7, label='Frontière entraînement / test')

plt.title('Prévision des passagers aériens avec un RNN')
plt.xlabel('Temps (mois)')
plt.ylabel('Nombre de passagers')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Afficher les prévisions récentes et les observations correspondantes.
print("\nPrévisions récentes et observations :")
for i in range(-24, 0):
    print(f"Mois {i} : observation={test_actual_denorm[i]:.0f}, prévision={test_predictions_denorm[i]:.0f}, erreur={abs(test_actual_denorm[i]-test_predictions_denorm[i]):.0f}")

## Questions à compléter

- Comment décririez-vous les prévisions du RNN ? Sont-elles satisfaisantes ?

... # À COMPLÉTER

- En examinant les erreurs du modèle, voyez-vous une tendance ou une cyclicité ?

... # À COMPLÉTER


## 2. Production mensuelle de bière en Australie

Téléchargez le jeu de données à cette adresse : https://github.com/ejgao/Time-Series-Datasets/blob/master/monthly-beer-production-in-austr.csv


In [ ]:
# Refaire la même analyse que pour les passagers aériens sur ce nouveau jeu de données.

... # À COMPLÉTER